# Qwen 2.5 Intent Distillation Pipeline v2
**Google Colab T4 GPU — Augmented Data Edition**

Changes from v1:
- **+110 augmented training samples** (compound intent, ASR noise, multi-product)
- **Improved system prompt** with priority rules for compound intents
- All T4/AMP fixes from v1 preserved

Pipeline:
1. Dataset Preparation (with augmented data)
2. Teacher Pseudo-Labeling (Qwen2.5-7B-Instruct, 4-bit)
3. Student QLoRA Fine-Tuning (Qwen2.5-0.5B-Instruct)
4. Evaluation (Accuracy / Macro-F1)
5. CPU Edge Benchmark (Latency / RAM)

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Đổi thư mục làm việc vào thư mục bạn muốn chứa dự án trên Drive
%cd /content/drive/MyDrive/

# 2. Clone repo từ GitHub về (Vì đây là repo Public nên bạn chỉ cần chạy URL bình thường)
!git clone https://github.com/kienng221105/Voice_Interaction_For_Robot.git

# 3. Di chuyển vào thư mục dự án
%cd Voice_Interaction_For_Robot


Mounted at /content/drive
/content/drive/MyDrive
fatal: destination path 'Voice_Interaction_For_Robot' already exists and is not an empty directory.
/content/drive/MyDrive/Voice_Interaction_For_Robot


In [2]:
!git pull

Already up to date.


In [ ]:
!pip install -q transformers==4.46.3 datasets accelerate peft==0.13.2 trl==0.12.2 bitsandbytes scikit-learn psutil tqdm

ERROR: Could not install packages due to an OSError: [Errno 107] Transport endpoint is not connected



## 1. Dataset Preparation (with Augmented Data)

In [ ]:
import json, random, os
from collections import Counter

os.makedirs('data/qwen_distill', exist_ok=True)
os.makedirs('reports', exist_ok=True)

gold_samples, unlabeled_samples = [], []
with open('data/raw/dataset_final.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        d = json.loads(line)
        if d.get('type') == 'unlabeled':
            unlabeled_samples.append({'text': d['text']})
        else:
            gold_samples.append({'text': d['text'], 'label': d['intent']})

# Load augmented training data (compound intent + ASR noise + multi-product)
augmented_samples = []
with open('data/raw/augmented_train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        augmented_samples.append(json.loads(line))
    with open('data/raw/augmented_train_v3.jsonl', 'r', encoding='utf-8') as f2:
        for line in f2:
            if not line.strip(): continue
            augmented_samples.append(json.loads(line))

hard_test_samples = []
with open('data/raw/dataset_150_bonus_v2.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        d = json.loads(line)
        hard_test_samples.append({'text': d['text'], 'label': d['intent']})

random.seed(42)
random.shuffle(gold_samples)
normal_test_samples = gold_samples[:50]
train_gold_samples  = gold_samples[50:]

# Merge original gold + augmented data
train_gold_augmented = train_gold_samples + augmented_samples

def save_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

save_jsonl(train_gold_augmented, 'data/qwen_distill/train_gold.jsonl')
save_jsonl(unlabeled_samples,    'data/qwen_distill/unlabeled.jsonl')
save_jsonl(normal_test_samples,  'data/qwen_distill/normal_test.jsonl')
save_jsonl(hard_test_samples,    'data/qwen_distill/hard_test.jsonl')

print(f'Train Gold (original): {len(train_gold_samples)}')
print(f'Augmented samples:     {len(augmented_samples)}')
print(f'Train Gold (total):    {len(train_gold_augmented)}')
print(f'Unlabeled:             {len(unlabeled_samples)}')
print(f'Normal Test:           {len(normal_test_samples)}')
print(f'Hard Test:             {len(hard_test_samples)}')

print('\n--- Augmented Train Intent Distribution ---')
for k, v in Counter(d['label'] for d in train_gold_augmented).most_common():
    print(f'  {k}: {v}')

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_627/60463941.py", line 4, in <cell line: 0>
    os.makedirs('data/qwen_distill', exist_ok=True)
  File "<frozen os>", line 215, in makedirs
  File "<frozen os>", line 225, in makedirs
OSError: [Errno 107] Transport endpoint is not connected: 'data'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'OSError' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/ultratb.

## 2. Improved System Prompt (with Priority Rules)

In [ ]:
SYSTEM_PROMPT = """You are an expert intent classifier for a Vietnamese vending machine assistant.
Analyze the user's input text and output ONLY the intent name.

The input may contain:
- ASR (speech recognition) noise: e.g. "bép si" = pepsi, "cô ka" = coca, "xtinh" = sting, "bảy áp" = 7up, "aqua fina" = aquafina
- Multiple products in one sentence
- Compound requests combining multiple actions

Valid Intents:
- greeting
- show_menu
- buy_product
- add_product
- change_product
- payment
- cancel
- help
- unknown

PRIORITY RULES (follow strictly):
1. If the sentence contains ANY payment keyword (thanh toán, trả tiền, quét mã, tính tiền) → always classify as "payment", even if it also mentions buying, adding, or changing products.
2. If the sentence mentions canceling one product AND buying/adding another (without payment) → classify as "cancel".
3. If the sentence only mentions buying multiple products → classify as "buy_product".

Examples:
Text: "cho tôi một lon coca" → buy_product
Text: "thêm pepsi rồi thanh toán luôn" → payment
Text: "đổi sang coca rồi trả tiền" → payment
Text: "cho bép si với cô ka" → buy_product
Text: "à thôi hủy đi" → cancel
"""

INTENTS = ['greeting','show_menu','buy_product','add_product',
           'change_product','payment','cancel','help','unknown']

def parse_intent(response_text):
    response_text = response_text.strip().lower()
    for intent in INTENTS:
        if intent in response_text:
            return intent
    return 'unknown'

## 3. Teacher Pseudo-Labeling (Qwen2.5-7B-Instruct)

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.notebook import tqdm

torch.cuda.empty_cache(); gc.collect()

TEACHER = 'Qwen/Qwen2.5-7B-Instruct'

bnb4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

t_tok = AutoTokenizer.from_pretrained(TEACHER)
t_model = AutoModelForCausalLM.from_pretrained(
    TEACHER, quantization_config=bnb4,
    device_map='auto', torch_dtype=torch.float16,
)

pseudo_labeled = []
for item in tqdm(unlabeled_samples, desc='Teacher labeling'):
    msgs = [
        {'role':'system','content': SYSTEM_PROMPT},
        {'role':'user',  'content': f'Text: "{item["text"]}"\nIntent:'},
    ]
    inp = t_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = t_tok([inp], return_tensors='pt').to(t_model.device)
    out = t_model.generate(**ids, max_new_tokens=10, do_sample=False)
    new_tokens = out[0][ids.input_ids.shape[1]:]
    resp = t_tok.decode(new_tokens, skip_special_tokens=True)
    pseudo_labeled.append({'text': item['text'], 'label': parse_intent(resp)})

save_jsonl(pseudo_labeled, 'data/qwen_distill/pseudo_labeled.jsonl')

# Merge: original gold + augmented + pseudo-labeled
train_distill = train_gold_augmented + pseudo_labeled
save_jsonl(train_distill, 'data/qwen_distill/train_distill.jsonl')
print(f'train_distill.jsonl: {len(train_distill)} samples')
print(f'  - Gold+Augmented: {len(train_gold_augmented)}')
print(f'  - Pseudo-labeled: {len(pseudo_labeled)}')

del t_model, t_tok; torch.cuda.empty_cache(); gc.collect()
print('Teacher model released.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

## 4. Student Fine-Tuning (QLoRA)
**Key fix:** `fp16=False, bf16=False` — disables AMP entirely.
The 0.5B model in 4-bit + LoRA uses < 2 GB VRAM, so fp32 training fits easily on T4.

In [ ]:
import torch, gc
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

torch.cuda.empty_cache(); gc.collect()

STUDENT = 'Qwen/Qwen2.5-0.5B-Instruct'

# Tokenizer
s_tok = AutoTokenizer.from_pretrained(STUDENT)
s_tok.pad_token = s_tok.eos_token

# Format dataset
rows = []
for item in train_distill:
    msgs = [
        {'role':'system',   'content': SYSTEM_PROMPT},
        {'role':'user',     'content': f'Text: "{item["text"]}"\nIntent:'},
        {'role':'assistant','content': item['label']},
    ]
    rows.append({'text': s_tok.apply_chat_template(msgs, tokenize=False)})
train_dataset = Dataset.from_list(rows)

# Load base model (4-bit)
bnb4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
s_model = AutoModelForCausalLM.from_pretrained(
    STUDENT, quantization_config=bnb4,
    device_map='auto', torch_dtype=torch.float16,
)

# Prepare for k-bit training
s_model = prepare_model_for_kbit_training(s_model)

# Apply LoRA
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    bias='none', task_type='CAUSAL_LM',
)
s_model = get_peft_model(s_model, lora_cfg)

# Nuclear fix: cast away any bfloat16 remnants
for name, param in s_model.named_parameters():
    if param.data.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
for name, buf in s_model.named_buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float16)

s_model.print_trainable_parameters()

# Training config — AMP OFF to prevent T4 BFloat16 crash
training_args = SFTConfig(
    dataset_text_field='text',
    output_dir='./student_qwen',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy='epoch',
    optim='paged_adamw_32bit',
    fp16=False,
    bf16=False,
    report_to='none',
)

trainer = SFTTrainer(
    model=s_model,
    train_dataset=train_dataset,
    processing_class=s_tok,
    args=training_args,
)

print(f'Starting student fine-tuning on {len(train_dataset)} samples...')
trainer.train()

trainer.model.save_pretrained('./student_qwen_adapter')
s_tok.save_pretrained('./student_qwen_adapter')
print('Adapter saved to ./student_qwen_adapter')

del trainer, s_model; torch.cuda.empty_cache(); gc.collect()

: 

## 5. Evaluation

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
import numpy as np

STUDENT = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER = './student_qwen_adapter'

def evaluate(model, tokenizer, data, label):
    y_true, y_pred = [], []
    for item in tqdm(data, desc=label):
        msgs = [
            {'role':'system','content': SYSTEM_PROMPT},
            {'role':'user',  'content': f'Text: "{item["text"]}"\nIntent:'},
        ]
        inp = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tokenizer([inp], return_tensors='pt').to(model.device)
        out = model.generate(**ids, max_new_tokens=10, do_sample=False)
        new_tokens = out[0][ids.input_ids.shape[1]:]
        resp = tokenizer.decode(new_tokens, skip_special_tokens=True)
        y_true.append(item['label'])
        y_pred.append(parse_intent(resp))
    
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f'\n{label}  Acc={acc:.4f}  Macro-F1={f1:.4f}')
    
    # Per-intent report
    labels_in_data = sorted(set(y_true + y_pred))
    print(classification_report(y_true, y_pred, labels=labels_in_data, zero_division=0))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels_in_data)
    print('Confusion Matrix:')
    header = ''.ljust(18) + '  '.join([l[:8].ljust(8) for l in labels_in_data])
    print(header)
    for i, row in enumerate(cm):
        print(labels_in_data[i][:18].ljust(18) + '  '.join([str(v).ljust(8) for v in row]))
    
    return acc, f1, y_true, y_pred

bnb4 = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
base = AutoModelForCausalLM.from_pretrained(
    STUDENT, quantization_config=bnb4,
    device_map='auto', torch_dtype=torch.float16,
)
eval_model = PeftModel.from_pretrained(base, ADAPTER)
eval_tok   = AutoTokenizer.from_pretrained(ADAPTER)

n_acc, n_f1, _, _ = evaluate(eval_model, eval_tok, normal_test_samples, 'Normal Test')
h_acc, h_f1, h_true, h_pred = evaluate(eval_model, eval_tok, hard_test_samples, 'Hard Test')

# Error analysis for hard test
print('\n=== HARD TEST ERROR ANALYSIS ===')
errors = [(t, p, hard_test_samples[i]['text']) 
          for i, (t, p) in enumerate(zip(h_true, h_pred)) if t != p]
print(f'Total errors: {len(errors)}/{len(hard_test_samples)}')
for true, pred, text in errors[:15]:
    print(f'  TRUE={true:18s} PRED={pred:18s} | {text}')

del eval_model, base; torch.cuda.empty_cache(); gc.collect()

: 

## 6. CPU Edge Benchmark

In [ ]:
import time, psutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

STUDENT = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER = './student_qwen_adapter'

def latency(model, tokenizer, text):
    msgs = [
        {'role':'system','content': SYSTEM_PROMPT},
        {'role':'user',  'content': f'Text: "{text}"\nIntent:'},
    ]
    inp = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer([inp], return_tensors='pt')
    t0 = time.time()
    _ = model.generate(**ids, max_new_tokens=10, do_sample=False)
    return time.time() - t0

t0 = time.time()
cpu_base  = AutoModelForCausalLM.from_pretrained(STUDENT, device_map='cpu', torch_dtype=torch.float32)
cpu_model = PeftModel.from_pretrained(cpu_base, ADAPTER)
cpu_tok   = AutoTokenizer.from_pretrained(ADAPTER)
load_s = time.time() - t0

mem_mb = psutil.Process().memory_info().rss / 1024**2
print(f'Load: {load_s:.1f}s | RAM: {mem_mb:.0f} MB')

latency(cpu_model, cpu_tok, 'warmup')
tests = [
    'mua 1 chai nước suối',
    'thôi không mua nữa',
    'thêm bép si rồi thanh toán luôn',
    'cho cô ka với xtinh nhé',
    'đổi sang aqua fina rồi trả tiền',
]
lats = [latency(cpu_model, cpu_tok, t) for t in tests]
print(f'Avg CPU latency: {sum(lats)/len(lats):.3f} s/query')
for t, l in zip(tests, lats):
    print(f'  {l:.3f}s | {t}')

: 